In [0]:
# Get all tables in skofy27.gold_dimensional schema
tables = spark.sql("SHOW TABLES IN skofy27.gold_dimensional").collect()

print(f"Found {len(tables)} tables in skofy27.gold_dimensional")
print("\nAltering tables to enable Uniform (Iceberg compatibility)...\n")

# Track results
success_count = 0
failed_tables = []

# Alter each table to enable Uniform with Iceberg
for table in tables:
    table_name = table.tableName
    full_table_name = f"skofy27.gold_dimensional.{table_name}"
    
    try:
        print(f"Processing: {full_table_name}")
        spark.sql(f"""
            ALTER TABLE {full_table_name}
            SET TBLPROPERTIES (
                'delta.universalFormat.enabledFormats' = 'iceberg'
            )
        """)
        print(f"✓ Successfully enabled Uniform for {full_table_name}\n")
        success_count += 1
    except Exception as e:
        print(f"✗ Failed to alter {full_table_name}: {str(e)}\n")
        failed_tables.append((full_table_name, str(e)))

# Summary
print("="*60)
print(f"Summary: {success_count}/{len(tables)} tables successfully altered")
if failed_tables:
    print(f"\nFailed tables ({len(failed_tables)}):")
    for table_name, error in failed_tables:
        print(f"  - {table_name}: {error}")